In [16]:
# Network Traffic Profiling and Attack Detection in Microservices Environments
## A PCAP-Based Analysis — Feature Extraction and Exploratory Data Analysis

This notebook extracts packet-level features from captured PCAP files across 
three normal traffic types and four attack types, then visualises the 
differences to support the dissertation results chapter.

**Traffic types:**
- `rest_normal` — Normal REST traffic to order-service (port 8080)
- `grpc_normal` — Normal gRPC traffic to inventory-service (port 9090)  
- `rabbitmq_normal` — Normal AMQP traffic to RabbitMQ (port 5672)
- `http_flood` — HTTP flood attack against order-service
- `slow_loris` — Slow Loris connection exhaustion attack
- `grpc_abuse` — gRPC stream abuse against inventory-service
- `queue_flood` — RabbitMQ queue flooding attack

SyntaxError: invalid character '—' (U+2014) (2200259524.py, line 9)

In [42]:
import sys
print(sys.executable)

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\python.exe


In [43]:
!pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
!{sys.executable} -m pip install pandas numpy matplotlib pyshark


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
import sys
!{sys.executable} -m pip install seaborn


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
!pip install nest_asyncio


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import nest_asyncio
nest_asyncio.apply()

import pyshark
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

print("All libraries loaded successfully")
print(f"pandas:      {pd.__version__}")
print(f"numpy:       {np.__version__}")
print(f"seaborn:     {sns.__version__}")

All libraries loaded successfully
pandas:      3.0.3
numpy:       2.4.6
seaborn:     0.13.2


In [48]:
## 1. Feature Extraction

Each PCAP file is loaded using PyShark. For every packet the following 
features are extracted:

| Feature | Description |
|---|---|
| `packet_length` | Total byte size of the packet |
| `inter_arrival_time` | Time delta since previous packet (seconds) |
| `dst_port` | TCP destination port |
| `src_port` | TCP source port |
| `tcp_flag_syn` | SYN flag set (1) or not (0) |
| `tcp_flag_fin` | FIN flag set (1) or not (0) |
| `tcp_flag_psh` | PSH flag set (1) or not (0) |
| `tcp_flag_ack` | ACK flag set (1) or not (0) |
| `traffic_type` | Label identifying the traffic type |
| `is_attack` | Binary target: 0 = normal, 1 = attack |

SyntaxError: invalid syntax (3118373350.py, line 3)

In [55]:
import os
import pyshark
import pandas as pd
import numpy as np

# ===============================
# PCAP FILE CONFIG
# ===============================
PCAP_FILES = [
    ('normal/rest_normal.pcapng',      'rest_normal',      0),
    ('normal/grpc_normal.pcapng',      'grpc_normal',      0),
    ('normal/rabbitmq_normal.pcapng',  'rabbitmq_normal',  0),
    ('attacks/http_flood.pcapng',      'http_flood',       1),
    ('attacks/slow_loris.pcapng',      'slow_loris',       1),
    ('attacks/grpc_abuse.pcapng',      'grpc_abuse',       1),
    ('attacks/queue_flood.pcapng',     'queue_flood',      1),
]

# ===============================
# FEATURE EXTRACTION FUNCTION
# ===============================
def extract_features(pcap_path, label, is_attack):

    if not os.path.exists(pcap_path):
        print(f"  ERROR: File not found: {pcap_path}")
        return pd.DataFrame()

    print(f"Processing: {pcap_path}")

    cap = pyshark.FileCapture(
        pcap_path,
        keep_packets=False,
        use_json=True
    )

    # 🔥 IMPORTANT FIX: force blocking load (avoids asyncio crash)
    packets = list(cap)

    records = []
    prev_time = None

    for pkt in packets:
        try:
            # Timestamp
            t = float(pkt.sniff_timestamp)

            # Inter-arrival time
            iat = (t - prev_time) if prev_time is not None else 0.0
            prev_time = t

            # TCP fields safely
            try:
                syn = int(pkt.tcp.flags_syn)
                fin = int(pkt.tcp.flags_fin)
                psh = int(pkt.tcp.flags_push)
                ack = int(pkt.tcp.flags_ack)
                dst_port = int(pkt.tcp.dstport)
                src_port = int(pkt.tcp.srcport)
            except AttributeError:
                syn = fin = psh = ack = 0
                dst_port = 0
                src_port = 0

            records.append({
                'timestamp': t,
                'packet_length': int(pkt.length),
                'inter_arrival_time': round(iat, 6),
                'dst_port': dst_port,
                'src_port': src_port,
                'tcp_flag_syn': syn,
                'tcp_flag_fin': fin,
                'tcp_flag_psh': psh,
                'tcp_flag_ack': ack,
                'traffic_type': label,
                'is_attack': is_attack
            })

        except AttributeError:
            pass

    cap.close()

    df = pd.DataFrame(records)
    print(f"  -> Extracted {len(df):,} packets from {label}")
    return df


# ===============================
# MAIN LOOP
# ===============================
print("=" * 50)
print("Starting feature extraction from all PCAP files")
print("=" * 50)

all_dfs = []

for path, label, is_attack in PCAP_FILES:
    df_temp = extract_features(path, label, is_attack)
    if not df_temp.empty:
        all_dfs.append(df_temp)

# ===============================
# COMBINE RESULTS
# ===============================
df = pd.concat(all_dfs, ignore_index=True)

print("\n" + "=" * 50)
print("EXTRACTION COMPLETE")
print(f"Total packets: {len(df):,}")
print("=" * 50)

print("\nPackets per traffic type:")
print(df['traffic_type'].value_counts().to_string())

print("\nClass balance (normal vs attack):")
print(df['is_attack'].value_counts().rename({0: 'Normal', 1: 'Attack'}).to_string())

Starting feature extraction from all PCAP files
Processing: normal/rest_normal.pcapng


RuntimeError: Cannot run the event loop while another loop is running

In [54]:
# Define all files — adjust paths if your folder structure differs
PCAP_FILES = [
    # path,                               label,              is_attack
    ('normal/rest_normal.pcapng',      'rest_normal',      0),
    ('normal/grpc_normal.pcapng',      'grpc_normal',      0),
    ('normal/rabbitmq_normal.pcapng',  'rabbitmq_normal',  0),
    ('attacks/http_flood.pcapng',      'http_flood',       1),
    ('attacks/slow_loris.pcapng',      'slow_loris',       1),
    ('attacks/grpc_abuse.pcapng',      'grpc_abuse',       1),
    ('attacks/queue_flood.pcapng',     'queue_flood',      1),
]

print("=" * 50)
print("Starting feature extraction from all PCAP files")
print("=" * 50)

all_dfs = []
for path, label, is_attack in PCAP_FILES:
    df_temp = extract_features(path, label, is_attack)
    if not df_temp.empty:
        all_dfs.append(df_temp)

# Combine into master DataFrame
df = pd.concat(all_dfs, ignore_index=True)

print()
print("=" * 50)
print(f"EXTRACTION COMPLETE")
print(f"Total packets: {len(df):,}")
print("=" * 50)
print()
print("Packets per traffic type:")
print(df['traffic_type'].value_counts().to_string())
print()
print("Class balance (normal vs attack):")
print(df['is_attack'].value_counts().rename({0: 'Normal', 1: 'Attack'}).to_string())

Starting feature extraction from all PCAP files
Processing: normal/rest_normal.pcapng


RuntimeError: Cannot run the event loop while another loop is running

In [21]:
## 2. Dataset Validation

Before analysis, we verify the extracted dataset is complete and 
contains no unexpected nulls or corrupted values.

SyntaxError: invalid syntax (4286791066.py, line 3)